# Keyword filter


In [2]:
from pathlib import Path
import json
import re

import pandas as pd

PROJECT_DIR = Path.cwd()
while PROJECT_DIR.name and not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "real_pii_collection" / "data"
INPUT_JSONL = DATA_DIR / "dvach_posts.jsonl"

OUTPUT_DIR = DATA_DIR / "outputs" / "keyword_filter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEYWORDS_TXT = OUTPUT_DIR / "curated_pii_keywords.txt"
CANDIDATES_JSONL = OUTPUT_DIR / f"{INPUT_JSONL.stem}.curated_keyword_candidates.jsonl"
SUMMARY_CSV = OUTPUT_DIR / f"{INPUT_JSONL.stem}.curated_keyword_summary.csv"

# 0 = весь файл; для теста можно поставить 10000.
MAX_ROWS = 0
MIN_TEXT_LEN = 20
MAX_TEXT_LEN = 5000

print("input:", INPUT_JSONL)
print("keywords_txt:", KEYWORDS_TXT)
print("candidates_jsonl:", CANDIDATES_JSONL)


input: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/dvach_posts.jsonl
keywords_txt: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/keyword_filter/curated_pii_keywords.txt
candidates_jsonl: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/keyword_filter/dvach_posts.curated_keyword_candidates.jsonl


In [2]:
# Curated keywords из YAML contexts, но без заведомо шумных слов.
# Можно спокойно редактировать руками: это не regex и не extractor gate.
KEYWORDS_BY_ENTITY = {
    "PHONE_NUMBER": [
        "номер телефона", "мой номер", "скинь номер", "дай номер", "кинь номер",
        "перезвони", "позвони", "позвоните", "звони", "телефон", "мобильный",
        "по номеру", "сбп", "система быстрых платежей",
    ],
    "EMAIL": [
        "email", "e-mail", "эл. почта", "электронная почта", "адрес почты",
        "напиши на почту", "почта", "мейл", "mail", "мыло",
    ],
    "IP_ADDRESS": ["ip-адрес", "айпишник", "айпи", "внешний ip"],

    "TELEGRAM": [
        "telegram", "телеграм", "телеграмм", "телега", "тг", "tg", "t.me", "tg://",
        "пиши в тг", "напиши в тг", "мой тг",
    ],
    "VK": ["vk.com", "vk.me", "vkontakte", "вконтакте", "в контакте", "мой вк"],
    "WHATSAPP": ["whatsapp", "ватсап", "вотсап", "вацап", "wa.me"],
    "DISCORD": ["discord", "дискорд", "discord.gg", "discord.com/users"],
    "INSTAGRAM": ["instagram", "инстаграм", "инста", "instagram.com"],
    "TWITTER_X": ["twitter.com", "x.com", "твиттер"],
    "TIKTOK": ["tiktok", "тикток", "tiktok.com"],
    "YOUTUBE": ["youtube.com/channel", "youtube.com/@", "ютуб канал", "мой канал"],
    "GITHUB": ["github.com", "гитхаб", "мой github"],
    "MAX": ["max", "max мессенджер"],

    "CAREER_PROFILE": [
        "hh.ru/resume", "резюме", "cv", "мой профиль", "ссылка на профиль",
        "career.habr", "habr career", "хабр карьера", "linkedin.com/in", "linkedin",
        "superjob", "авито работа",
    ],

    "PASSPORT_RF": ["паспорт", "паспортные данные", "серия паспорта", "номер паспорта"],
    "PASSPORT_INTERNATIONAL": ["загранпаспорт", "загран", "биометрический загранпаспорт"],
    "DRIVER_LICENSE": ["водительское удостоверение", "водительские права"],
    "SNILS": ["снилс", "страховой номер", "индивидуальный лицевой счет", "индивидуальный лицевой счёт"],
    "INN": ["инн", "идентификационный номер налогоплательщика", "налоговый номер"],
    "MILITARY_ID": ["военный билет", "военник"],
    "OMS_DMS": ["полис омс", "омс", "полис дмс", "дмс", "медицинский полис"],
    "RESIDENCE_PERMIT": ["вид на жительство", "внж"],
    "WORK_BOOK": ["трудовая книжка", "электронная трудовая", "стд-р", "стд-пфр"],

    "BANK_CARD": ["номер карты", "банковская карта", "дебетовая карта", "кредитная карта"],
    "BANK_ACCOUNT": ["расчетный счет", "расчетный счёт", "банковский счет", "банковский счёт", "лицевой счет", "лицевой счёт"],
    "PAYMENT_SYSTEMS": ["qiwi", "киви", "юmoney", "юмани", "webmoney", "wmid", "яндекс деньги"],
    "CRYPTO_WALLET": ["криптокошелек", "криптокошелёк", "crypto wallet", "bitcoin address", "btc address", "ethereum address", "usdt"],
    "IBAN_SWIFT": ["iban", "swift", "bic", "бик"],

    "TRANSPORT": ["vin", "вин номер", "госномер", "гос номер", "номер птс", "номер стс", "полис осаго", "полис каско"],
    "TRACKING": ["трек-номер", "трек номер", "почта россии", "ems", "s10"],
    "COURT": ["дело №", "номер дела", "номер судебного приказа", "судебный приказ", "исполнительное производство", "арбитражное дело"],

    "EDUCATION_DOC": ["номер диплома", "серия диплома", "номер аттестата", "студенческий билет"],
    "FAMILY_DOC": ["свидетельство о рождении", "свидетельство о браке", "свидетельство о расторжении брака"],
    "CRIMINAL_RECORD_CERTIFICATE": ["справка о судимости", "справка об отсутствии судимости"],
}

# Дополнительные сильные строковые сигналы, которые не завязаны на конкретный config context.
EXTRA_KEYWORDS = {
    "AT_HANDLE": ["@"],
    "URL_PROFILE_HINT": ["/resume/", "/users/", "/people/", "/id", "id"],
}

def norm(value: str) -> str:
    return " ".join(str(value).strip().lower().split())

keyword_to_entities = {}
for entity, words in {**KEYWORDS_BY_ENTITY, **EXTRA_KEYWORDS}.items():
    for word in words:
        word = norm(word)
        if word:
            keyword_to_entities.setdefault(word, set()).add(entity)

keywords = sorted(keyword_to_entities, key=lambda x: (-len(x), x))

with KEYWORDS_TXT.open("w", encoding="utf-8") as f:
    for keyword in keywords:
        f.write(f"{keyword}\t{', '.join(sorted(keyword_to_entities[keyword]))}\n")

print("keywords:", len(keywords))
for kw in keywords[:80]:
    print(repr(kw), "=>", sorted(keyword_to_entities[kw]))


keywords: 176
'идентификационный номер налогоплательщика' => ['INN']
'свидетельство о расторжении брака' => ['FAMILY_DOC']
'справка об отсутствии судимости' => ['CRIMINAL_RECORD_CERTIFICATE']
'биометрический загранпаспорт' => ['PASSPORT_INTERNATIONAL']
'индивидуальный лицевой счет' => ['SNILS']
'индивидуальный лицевой счёт' => ['SNILS']
'исполнительное производство' => ['COURT']
'водительское удостоверение' => ['DRIVER_LICENSE']
'свидетельство о рождении' => ['FAMILY_DOC']
'система быстрых платежей' => ['PHONE_NUMBER']
'номер судебного приказа' => ['COURT']
'свидетельство о браке' => ['FAMILY_DOC']
'электронная трудовая' => ['WORK_BOOK']
'youtube.com/channel' => ['YOUTUBE']
'справка о судимости' => ['CRIMINAL_RECORD_CERTIFICATE']
'водительские права' => ['DRIVER_LICENSE']
'студенческий билет' => ['EDUCATION_DOC']
'discord.com/users' => ['DISCORD']
'вид на жительство' => ['RESIDENCE_PERMIT']
'медицинский полис' => ['OMS_DMS']
'паспортные данные' => ['PASSPORT_RF']
'ссылка на профиль' =>

In [3]:
def keyword_in_text(keyword: str, text_lower: str) -> bool:
    # Для коротких ключей типа тг/tg/ip/id/@ нужны границы, иначе будет много мусора.
    if keyword == "@":
        return "@" in text_lower
    if len(keyword) <= 3 and keyword.isalnum():
        return re.search(rf"(?<![\w@]){re.escape(keyword)}(?![\w@])", text_lower) is not None
    return keyword in text_lower


def find_keywords(text: str) -> list[str]:
    text_lower = text.lower()
    return [keyword for keyword in keywords if keyword_in_text(keyword, text_lower)]


In [4]:
processed = 0
kept = 0
keyword_counts = {}
entity_counts = {}

with INPUT_JSONL.open("r", encoding="utf-8") as src, CANDIDATES_JSONL.open("w", encoding="utf-8") as dst:
    for line_idx, line in enumerate(src):
        if MAX_ROWS and processed >= MAX_ROWS:
            break
        if not line.strip():
            continue
        processed += 1
        row = json.loads(line)
        text = str(row.get("text") or "")
        if not (MIN_TEXT_LEN <= len(text) <= MAX_TEXT_LEN):
            continue

        matched_keywords = find_keywords(text)
        if not matched_keywords:
            continue

        matched_entities = sorted({entity for kw in matched_keywords for entity in keyword_to_entities[kw]})
        out = dict(row)
        out["source_line_idx"] = line_idx
        out["matched_keywords"] = matched_keywords
        out["matched_entities"] = matched_entities
        dst.write(json.dumps(out, ensure_ascii=False) + "\n")
        kept += 1

        for kw in matched_keywords:
            keyword_counts[kw] = keyword_counts.get(kw, 0) + 1
        for entity in matched_entities:
            entity_counts[entity] = entity_counts.get(entity, 0) + 1

summary_rows = []
for kw, count in sorted(keyword_counts.items(), key=lambda x: (-x[1], x[0])):
    summary_rows.append({"kind": "keyword", "value": kw, "count": count, "entities": ", ".join(sorted(keyword_to_entities[kw]))})
for entity, count in sorted(entity_counts.items(), key=lambda x: (-x[1], x[0])):
    summary_rows.append({"kind": "entity", "value": entity, "count": count, "entities": ""})

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

print("processed:", processed)
print("kept:", kept)
print("conversion:", round(kept / processed * 100, 2) if processed else 0, "%")
print("top keywords:")
for kw, count in sorted(keyword_counts.items(), key=lambda x: (-x[1], x[0]))[:50]:
    print(count, repr(kw), sorted(keyword_to_entities[kw]))
print("top entities:")
for entity, count in sorted(entity_counts.items(), key=lambda x: (-x[1], x[0]))[:50]:
    print(count, entity)
print("saved:", CANDIDATES_JSONL)
print("summary:", SUMMARY_CSV)


processed: 127554
kept: 6569
conversion: 5.15 %
top keywords:
2385 'телефон' ['PHONE_NUMBER']
764 '@' ['AT_HANDLE']
566 'звони' ['PHONE_NUMBER']
430 'тг' ['TELEGRAM']
325 'резюме' ['CAREER_PROFILE']
184 'телеграм' ['TELEGRAM']
184 'тикток' ['TIKTOK']
183 'паспорт' ['PASSPORT_RF']
176 'github.com' ['GITHUB']
170 'гитхаб' ['GITHUB']
169 'max' ['MAX']
160 'позвони' ['PHONE_NUMBER']
143 'инста' ['INSTAGRAM']
128 'мыло' ['EMAIL']
118 'дискорд' ['DISCORD']
108 'id' ['URL_PROFILE_HINT']
96 'мобильный' ['PHONE_NUMBER']
95 'айпи' ['IP_ADDRESS']
95 'загран' ['PASSPORT_INTERNATIONAL']
81 'телега' ['TELEGRAM']
67 'твиттер' ['TWITTER_X']
59 'mail' ['EMAIL']
59 'telegram' ['TELEGRAM']
56 'мейл' ['EMAIL']
54 'инстаграм' ['INSTAGRAM']
54 'почта' ['EMAIL']
53 'vk.com' ['VK']
50 't.me' ['TELEGRAM']
46 'ватсап' ['WHATSAPP']
45 'киви' ['PAYMENT_SYSTEMS']
44 'youtube.com/@' ['YOUTUBE']
41 'телеграмм' ['TELEGRAM']
38 'дмс' ['OMS_DMS']
34 'вконтакте' ['VK']
34 'внж' ['RESIDENCE_PERMIT']
34 'перезвони' ['PHON

In [5]:
# Превью первых 50 кандидатов без вывода всего jsonl.
preview_rows = []
with CANDIDATES_JSONL.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 50:
            break
        row = json.loads(line)
        preview_rows.append({
            "i": i,
            "board": row.get("board"),
            "keywords": ", ".join(row.get("matched_keywords", [])[:10]),
            "entities": ", ".join(row.get("matched_entities", [])[:10]),
            "text": str(row.get("text", ""))[:350].replace("\n", " "),
        })

pd.DataFrame(preview_rows)


,i,board,keywords,entities,text
0,0,wrk,военник,MILITARY_ID,>>2948564 Отопление рисовал в автокаде. Делал...
1,1,wrk,телефон,PHONE_NUMBER,Нет ИТ не выходит А я могу РАБотать только за ...
2,2,wrk,резюме,CAREER_PROFILE,">>2958982 Дак накрути 3 года опыта в резюме, ..."
3,3,wrk,тикток,TIKTOK,>>3138119 >Но там интернетиков не будет и под...
4,4,wrk,загран,PASSPORT_INTERNATIONAL,">>3230793 >Почему пенсия не сделал, раз шиза?..."
5,5,wrk,телефон,PHONE_NUMBER,Другой тревожник на связи. Решил как кабан беж...
6,6,wrk,"позвони, звони",PHONE_NUMBER,>>3272599 Позвони знакомому
7,7,wrk,тикток,TIKTOK,>>3310293 Если только ты имеешь ввиду опцию с...
8,8,wrk,"телефон, звони",PHONE_NUMBER,>>3311423 Вспомните пасту с хабра. Чел технар...
9,9,wrk,"позвони, звони",PHONE_NUMBER,ДНО-тред #1104 - Устал от нищеты Тред о выжива...


## Быстрый ad-hoc фильтр по списку слов/regex

Эта ячейка не использует основной словарь. Просто вписываешь список строк в `ADHOC_KEYWORDS` и/или regex в `ADHOC_REGEXES`, запускаешь и получаешь отдельный jsonl. Удобно для разовых поисков вроде начал карт МИР `2200..2204`.


In [3]:
# Быстрый разовый фильтр. Можно менять прямо тут.
ADHOC_NAME = "mir_card_2200_2204"

# Простые substring keywords. Для МИР лучше оставить пустым и использовать regex ниже,
# потому что "2200" само по себе может давать мусор.
ADHOC_KEYWORDS = [
    # "2200", "2201", "2202", "2203", "2204",
]

# Regex-паттерны. Для начала карт МИР 2200-2204 + еще 12 цифр.
ADHOC_REGEXES = [
    r"(?<!\d)(?:2200|2201|2202|2203|2204)[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}(?!\d)",
]

ADHOC_OUTPUT_JSONL = OUTPUT_DIR / f"{INPUT_JSONL.stem}.{ADHOC_NAME}.jsonl"
ADHOC_SUMMARY_TXT = OUTPUT_DIR / f"{INPUT_JSONL.stem}.{ADHOC_NAME}.summary.txt"

compiled_regexes = [re.compile(pattern, re.I) for pattern in ADHOC_REGEXES]
adhoc_keywords_norm = [str(keyword).lower() for keyword in ADHOC_KEYWORDS if str(keyword).strip()]

processed = 0
kept = 0
keyword_hits = {keyword: 0 for keyword in adhoc_keywords_norm}
regex_hits = {pattern: 0 for pattern in ADHOC_REGEXES}

with INPUT_JSONL.open("r", encoding="utf-8") as src, ADHOC_OUTPUT_JSONL.open("w", encoding="utf-8") as dst:
    for line_idx, line in enumerate(src):
        if MAX_ROWS and processed >= MAX_ROWS:
            break
        if not line.strip():
            continue
        processed += 1
        row = json.loads(line)
        text = str(row.get("text") or "")
        if not (MIN_TEXT_LEN <= len(text) <= MAX_TEXT_LEN):
            continue

        text_lower = text.lower()
        matched_keywords = [keyword for keyword in adhoc_keywords_norm if keyword in text_lower]
        matched_regexes = []
        for pattern, regex in zip(ADHOC_REGEXES, compiled_regexes):
            if regex.search(text):
                matched_regexes.append(pattern)

        if not matched_keywords and not matched_regexes:
            continue

        for keyword in matched_keywords:
            keyword_hits[keyword] += 1
        for pattern in matched_regexes:
            regex_hits[pattern] += 1

        out = dict(row)
        out["source_line_idx"] = line_idx
        out["adhoc_filter_name"] = ADHOC_NAME
        out["matched_adhoc_keywords"] = matched_keywords
        out["matched_adhoc_regexes"] = matched_regexes
        dst.write(json.dumps(out, ensure_ascii=False) + "\n")
        kept += 1

summary_lines = [
    f"input={INPUT_JSONL}",
    f"output={ADHOC_OUTPUT_JSONL}",
    f"processed={processed}",
    f"kept={kept}",
    f"conversion={(kept / processed * 100):.4f}%" if processed else "conversion=0%",
    "",
    "keyword_hits:",
]
summary_lines += [f"{count}\t{keyword}" for keyword, count in sorted(keyword_hits.items(), key=lambda x: (-x[1], x[0]))]
summary_lines += ["", "regex_hits:"]
summary_lines += [f"{count}\t{pattern}" for pattern, count in sorted(regex_hits.items(), key=lambda x: (-x[1], x[0]))]
ADHOC_SUMMARY_TXT.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print("processed:", processed)
print("kept:", kept)
print("output:", ADHOC_OUTPUT_JSONL)
print("summary:", ADHOC_SUMMARY_TXT)
print("keyword_hits:", keyword_hits)
print("regex_hits:", regex_hits)


processed: 127554
kept: 0
output: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/keyword_filter/dvach_posts.mir_card_2200_2204.jsonl
summary: /Users/artemzmailov/Desktop/kitoboy-PII/real_pii_collection/data/outputs/keyword_filter/dvach_posts.mir_card_2200_2204.summary.txt
keyword_hits: {}
regex_hits: {'(?<!\\d)(?:2200|2201|2202|2203|2204)[\\s-]?\\d{4}[\\s-]?\\d{4}[\\s-]?\\d{4}(?!\\d)': 0}
